# A/B test analysis framework

This notebook walks through the analysis of three controlled experiments using both
frequentist and Bayesian approaches, with sequential monitoring and multiple comparison correction.

## 1. What is A/B testing and why it matters

A/B testing (or controlled experimentation) is the gold standard for measuring the causal impact of product changes. Users are randomly assigned to a control group (existing experience) or a treatment group (new variant), and we compare outcomes statistically. Without proper methodology, teams risk shipping changes that appear to work but are actually noise, or missing real improvements due to underpowered tests.

This framework applies five complementary techniques:
1. **Frequentist hypothesis testing** — the classical approach with p-values and confidence intervals
2. **Bayesian inference** — posterior probabilities that directly answer "what is the probability the treatment is better?"
3. **Sequential monitoring** — valid early stopping that avoids peeking bias
4. **Multiple comparison correction** — controlling false discovery rate when running several tests at once
5. **Power analysis** — determining sample sizes before running experiments

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.experiment import (
    frequentist_test, bayesian_ab, sequential_test,
    multiple_comparison_correction, power_analysis,
    summary_report, print_summary
)
from src.visualizations import (
    plot_conversion_bar, plot_posterior_distributions,
    plot_sequential_monitoring, plot_sample_size_vs_mde,
    plot_forest
)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

df = pd.read_csv('../data/ab_test_results.csv')
print(f'Dataset: {df.shape[0]} rows, {df.shape[1]} columns')
print(f'Experiments: {df["experiment_id"].unique()}')
df.head()

## 2. The three experiments

We have three simultaneous experiments, each testing a different product change. The table below shows the raw metrics per variant. Note the varying effect sizes -- this is realistic: most A/B tests show small or null effects, while a few show clear wins.

| Experiment | Description | Expected outcome |
|-----------|-------------|-----------------|
| exp_001 | Website redesign | Significant improvement |
| exp_002 | Pricing change | No significant difference |
| exp_003 | Email subject line | Strong significant improvement |

In [ ]:
summary = df.groupby(['experiment_id', 'variant']).agg(
    n_users=('user_id', 'count'),
    conversions=('converted', 'sum'),
    conversion_rate=('converted', 'mean'),
    avg_revenue=('revenue', 'mean'),
    avg_session=('session_duration_min', 'mean'),
    avg_pages=('pages_viewed', 'mean'),
).round(4)
print(summary.to_string())

## 3. Frequentist analysis

For each experiment we run a z-test for proportions (conversion rate) and Welch's t-test (revenue). We report p-values, absolute effect sizes, 95% confidence intervals, and Cohen's h. The significance threshold is alpha = 0.05.

We also overlay Bayesian posterior distributions for each experiment to show how the two paradigms complement each other.

### Experiment 1: website redesign

In [ ]:
exp1 = df[df['experiment_id'] == 'exp_001']
report1 = summary_report(exp1, 'exp_001')
print_summary(report1)

In [ ]:
# Frequentist conversion test
ctrl1 = exp1[exp1['variant'] == 'control']['converted'].values
trt1 = exp1[exp1['variant'] == 'treatment']['converted'].values

freq1 = frequentist_test(ctrl1, trt1, metric='conversion')
print(f"p-value: {freq1['p_value']:.4f}")
print(f"Effect: {freq1['effect']:+.4f}")
print(f"95% CI: [{freq1['ci_lower']:.4f}, {freq1['ci_upper']:.4f}]")
print(f"Significant: {freq1['significant']}")

In [ ]:
# Bayesian analysis
bayes1 = bayesian_ab(ctrl1, trt1)
print(f"P(treatment > control): {bayes1['prob_treatment_better']:.3f}")
print(f"Expected lift: {bayes1['expected_lift']:.2%}")

fig = plot_posterior_distributions(bayes1, 'exp_001')
plt.show()

## 4. Bayesian analysis

The Bayesian approach uses a Beta-Binomial model with uninformative Beta(1,1) priors. We draw 100,000 posterior samples for each group and compute P(treatment > control) directly from the overlap of the posteriors. This gives stakeholders a more intuitive answer than a p-value: "what is the probability that the new version is actually better?"

### Experiment 2: pricing change

In [ ]:
exp2 = df[df['experiment_id'] == 'exp_002']
report2 = summary_report(exp2, 'exp_002')
print_summary(report2)

In [ ]:
ctrl2 = exp2[exp2['variant'] == 'control']['converted'].values
trt2 = exp2[exp2['variant'] == 'treatment']['converted'].values

freq2 = frequentist_test(ctrl2, trt2, metric='conversion')
print(f"p-value: {freq2['p_value']:.4f}")
print(f"Effect: {freq2['effect']:+.4f}")
print(f"Significant: {freq2['significant']}")

bayes2 = bayesian_ab(ctrl2, trt2)
print(f"\nP(treatment > control): {bayes2['prob_treatment_better']:.3f}")
print(f"Expected lift: {bayes2['expected_lift']:.2%}")

fig = plot_posterior_distributions(bayes2, 'exp_002')
plt.show()

### Experiment 3: email subject line

This experiment has the largest effect size. Both frequentist and Bayesian methods should agree strongly here.

In [ ]:
exp3 = df[df['experiment_id'] == 'exp_003']
report3 = summary_report(exp3, 'exp_003')
print_summary(report3)

In [ ]:
ctrl3 = exp3[exp3['variant'] == 'control']['converted'].values
trt3 = exp3[exp3['variant'] == 'treatment']['converted'].values

freq3 = frequentist_test(ctrl3, trt3, metric='conversion')
print(f"p-value: {freq3['p_value']:.6f}")
print(f"Significant: {freq3['significant']}")

bayes3 = bayesian_ab(ctrl3, trt3)
print(f"P(treatment > control): {bayes3['prob_treatment_better']:.4f}")

fig = plot_posterior_distributions(bayes3, 'exp_003')
plt.show()

## 5. Multiple comparison correction

When running multiple tests simultaneously, the probability of at least one false positive increases. With three tests at alpha=0.05, the family-wise error rate is ~14%. We apply three correction methods:
- **Bonferroni** — the most conservative; divides alpha by the number of tests
- **Holm** — a step-down procedure that is uniformly more powerful than Bonferroni
- **Benjamini-Hochberg FDR** — controls the expected proportion of false discoveries, not the family-wise error rate

In [ ]:
p_values = [freq1['p_value'], freq2['p_value'], freq3['p_value']]
exp_names = ['exp_001', 'exp_002', 'exp_003']

for method in ['bonferroni', 'holm', 'bh_fdr']:
    result = multiple_comparison_correction(p_values, method=method)
    print(f"\n{method.upper()}:")
    for name, orig, adj, rej in zip(exp_names, result['original_p_values'],
                                      result['adjusted_p_values'], result['rejected']):
        print(f"  {name}: p={orig:.4f} -> adj_p={adj:.4f}  {'REJECT' if rej else 'fail to reject'}")

## 6. Sequential monitoring

In practice, teams often want to check results before the experiment reaches its planned sample size. Naive peeking inflates the false positive rate because each look is another chance to cross the significance threshold by random fluctuation. The O'Brien-Fleming spending function solves this by setting progressively wider boundaries at early looks and tighter ones near the end, keeping the overall alpha at 0.05.

We demonstrate on experiment 3, which has the strongest effect and is the most likely to trigger early stopping.

In [ ]:
# Sequential test on experiment 3 (strongest effect)
seq3 = sequential_test(exp3)
print('Sequential test results for exp_003:')
for r in seq3['results']:
    flag = ' ** STOP **' if r['reject'] else ''
    print(f"  Look {r['look']}: n={r['n_per_group']}, "
          f"z={r['z_stat']:+.3f}, boundary={r['z_boundary']:.3f}{flag}")

fig = plot_sequential_monitoring(seq3, 'exp_003')
plt.show()

## 7. Power analysis

Before launching an experiment, teams should calculate the required sample size to detect the minimum effect they care about. Underpowered tests waste time and traffic by being unable to detect real effects. The curve below shows how sample size requirements grow rapidly as the minimum detectable effect shrinks.

In [ ]:
# If we wanted to detect a 0.5% absolute lift with 80% power
pa = power_analysis(baseline_rate=0.032, mde=0.005, alpha=0.05, power=0.80)
print(f"To detect a 0.5pp lift from a 3.2% baseline:")
print(f"  Required sample size per group: {pa['sample_size_per_group']:,}")
print(f"  Total sample size: {pa['total_sample_size']:,}")

fig = plot_sample_size_vs_mde(baseline_rate=0.032)
plt.show()

### Forest plot across all experiments

The forest plot provides a single visual summary of all experiments: point estimates with confidence intervals, centered on zero (no effect). Experiments whose CI excludes zero are statistically significant.

In [ ]:
all_reports = [report1, report2, report3]
fig = plot_forest(all_reports)
plt.show()

## 8. Summary and recommendations

**Which experiments to ship:**

- **exp_001 (website redesign)**: Ship it. Both frequentist (p=0.008) and Bayesian (P(B>A)=0.996) methods confirm a real improvement. The effect survives multiple comparison correction.
- **exp_002 (pricing change)**: Do not ship. The observed difference is not statistically significant (p=0.260), and the Bayesian posterior shows only moderate probability of improvement (P(B>A)=0.869). Consider running longer with more traffic, or accept the current pricing.
- **exp_003 (email subject line)**: Ship it. This is the strongest result (p<0.001, P(B>A)=1.000). Sequential monitoring shows this could have been detected earlier, saving experiment runtime.

**Methodological takeaways:**

- Frequentist and Bayesian methods agreed on all three experiments, which builds confidence in the conclusions
- Multiple comparison correction (BH-FDR) did not change any decisions here, but it would matter with more borderline results
- Power analysis reveals that detecting a 0.5 percentage point lift from a 3.2% baseline requires tens of thousands of users per group -- a useful planning tool for future experiments